In [ ]:
"""
Program Name: AudioRecognition.py
Description: It reads sounds from .wav files.
Programmer(s): Sam Harrison, Naran Bat
Date Made: 3/29/25
Date(s) Revised: 3/30/25 - Added method to determine direction and distance of sound source
Preconditions: 
Postconditions: 
Errors/Exceptions:
Side Effects:
Invariants: 
Known Faults:
"""

import speech_recognition as sr

# initializes recognizer
recognizer = sr.Recognizer()

# the test file I'm using
audio_file = "test.wav"

def get_audio_direction_and_distance(audio_file):
    """Determines the direction and distance of the sound source using time difference of arrival (TDOA)."""
    with wave.open(audio_file, 'rb') as wav:
        n_channels = wav.getnchannels() # Number of channels
        if n_channels != 2:
            raise ValueError("Audio file must have two channels for triangulation.")
        
        framerate = wav.getframerate() # Sampling frequency
        frames = wav.readframes(-1) # Read all frames
        audio_data = np.frombuffer(frames, dtype=np.int16) # Assuming 16-bit PCM format
        
        left_channel = audio_data[0::2]  # Extract left channel
        right_channel = audio_data[1::2]  # Extract right channel
        
        # Cross-correlation to find time difference
        correlation = np.correlate(left_channel, right_channel, mode='full') # Cross-correlation
        delay = np.argmax(correlation) - (len(left_channel) - 1) # Delay in samples
        time_diff = delay / framerate # Convert to seconds
        
        # Assuming microphone spacing is 0.2m and speed of sound is 343m/s
        mic_distance = 0.2
        speed_of_sound = 343
        
        angle = np.arcsin((time_diff * speed_of_sound) / mic_distance) # Calculate angle in radians
        
        # Estimating distance using amplitude attenuation (simplified model)
        amplitude_left = np.max(np.abs(left_channel)) # Max amplitude of left channel
        amplitude_right = np.max(np.abs(right_channel)) # Max amplitude of right channel
        avg_amplitude = (amplitude_left + amplitude_right) / 2 # Average amplitude
        reference_amplitude = 32768  # Max possible value for 16-bit audio
        distance = (reference_amplitude / max(avg_amplitude, 1)) * 0.5  # Scaled arbitrary reference
        
        return angle, distance

with sr.AudioFile(audio_file) as source:
    print("Reading audio file...")
    audio_data = recognizer.record(source)

    try:
        text = recognizer.recognize_google(audio_data)
        print("You said:", text)
    except sr.UnknownValueError:
        print("Sorry, I couldn't understand the audio.")
    except sr.RequestError as e:
        print(f"Could not request results; {e}")


Reading audio file...
You said: Slide the box into that empty space the plant grew large and green in the window the beam dropped down on the workman's head pink clouds floated with the breeze she danced like a swan tall and graceful the tube was blown and the tire flat and useless it is late in the morning on the old wall clocks the last switch cannot be turned off the fight will end in just six minutes
